# TextTovoz TTS Pipeline

**Personal use only. AI-generated audio. Do not redistribute.**

In [ ]:
import importlib
import logging
import subprocess
import sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")
logger = logging.getLogger("texttovoz.notebook")

min_python = (3, 10)
if sys.version_info[:2] < min_python:
    logger.error(
        "Python 3.10+ is required. Runtime is %s",
        sys.version.split()[0],
    )
    raise RuntimeError("Python 3.10+ is required for TextTovoz.")

# 1. Clone the texttovoz repo so the local package can be imported.
#    Pin the branch that contains the pipeline implementation.
#    After the PR is merged to main, change REPO_BRANCH to "main".
REPO_URL = "https://github.com/CristianMz21/textTovoz.git"
REPO_BRANCH = "feat/texttovoz-tts-pipeline"
REPO_DIR = Path("/content/textTovoz")

if not (REPO_DIR / "src" / "texttovoz" / "__init__.py").exists():
    logger.info("Cloning %s @ %s into %s", REPO_URL, REPO_BRANCH, REPO_DIR)
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth=1",
            "-b",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ]
    )
else:
    logger.info("Reusing existing clone at %s", REPO_DIR)

# 2. Install the local package editable so subsequent cells can
#    `import texttovoz`.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)]
)

# 3. Install Colab runtime dependencies.
packages = [
    "chatterbox-tts==0.1.7",
    "torch",
    "torchaudio",
    "soundfile",
    "pydantic",
    "pyyaml",
    "tqdm",
    "ipython",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

torch = importlib.import_module("torch")
if not torch.cuda.is_available():
    logger.error("No GPU detected. In Colab, choose a T4 GPU runtime.")
    raise RuntimeError("A CUDA GPU is required for Chatterbox TTS.")

logger.info(
    "Dependency check passed with Python %s and GPU %s",
    sys.version.split()[0],
    torch.cuda.get_device_name(0),
)

In [ ]:
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HOME"] = "/content/.cache/huggingface"
model_id = "ResembleAI/Chatterbox-Multilingual-es-mx-latam"

# Self-heal: if the package is not importable (stale Colab cache
# or older notebook revision), clone + editable-install the repo
# right here. This makes the cell robust on its own.
if importlib.util.find_spec("texttovoz") is None:
    REPO_URL = "https://github.com/CristianMz21/textTovoz.git"
    REPO_BRANCH = "feat/texttovoz-tts-pipeline"
    REPO_DIR = Path("/content/textTovoz")
    if not (REPO_DIR / "src" / "texttovoz" / "__init__.py").exists():
        logger.info(
            "Cloning %s @ %s into %s", REPO_URL, REPO_BRANCH, REPO_DIR
        )
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth=1",
                "-b",
                REPO_BRANCH,
                REPO_URL,
                str(REPO_DIR),
            ]
        )
    else:
        logger.info("Reusing existing clone at %s", REPO_DIR)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            str(REPO_DIR),
        ]
    )
    importlib.invalidate_caches()
    logger.info("texttovoz package installed.")

hf_hub = importlib.import_module("huggingface_hub")
logger.info("Pre-warming Hugging Face cache at %s", os.environ["HF_HOME"])
hf_hub.snapshot_download(repo_id=model_id)

tts_module = importlib.import_module("texttovoz.tts")
if not tts_module.is_available():
    raise RuntimeError("Chatterbox import check failed after installation.")
logger.info("Hugging Face cache and Chatterbox import check completed.")

In [ ]:
from pathlib import Path

from texttovoz import config, manifest, pipeline

cfg = config.TTSConfig(
    input_path=Path("/content/subtitle.txt"),
    chunks_dir=Path("/content/out/chunks"),
    output_dir=Path("/content/out"),
    manifest_path=Path("/content/out/chunks/manifest.jsonl"),
    output_wav_path=Path("/content/out/full.wav"),
    language_id="es",
)
logger.info(
    "Configured TextTovoz with manifest schema %s",
    manifest.ChunkRecord.__name__,
)
cfg

## Upload transcript

Upload a file named `subtitle.txt`; it will be saved to `/content/subtitle.txt`.

In [ ]:
from google.colab import files

uploaded = files.upload()
if "subtitle.txt" not in uploaded:
    raise RuntimeError("Please upload a file named subtitle.txt.")

Path("/content/subtitle.txt").write_bytes(uploaded["subtitle.txt"])
logger.info("Saved transcript to /content/subtitle.txt")

In [ ]:
from dataclasses import replace

from IPython.display import Audio, display

preview_cfg = replace(
    cfg,
    output_dir=Path("/content/out/preview"),
    chunks_dir=Path("/content/out/preview/chunks"),
    manifest_path=Path("/content/out/preview/chunks/manifest.jsonl"),
    output_wav_path=Path("/content/out/preview/full.wav"),
    to_chunk=2,
)
logger.info(
    "Generating the first %s chunks for a prosody preview.",
    preview_cfg.to_chunk,
)
preview_result = pipeline.run(preview_cfg)
logger.info(
    "Preview generated=%s skipped=%s errors=%s",
    preview_result.generated,
    preview_result.skipped,
    preview_result.errors,
)
display(Audio(str(preview_result.output_wav_path)))

In [ ]:
import tqdm

logger.info("Starting full TextTovoz run; package logs chunk progress.")
with tqdm.tqdm(total=1, desc="TextTovoz full pipeline") as progress:
    result = pipeline.run(cfg)
    progress.update(1)

logger.info(
    "Full run complete: total=%s selected=%s generated=%s skipped=%s errors=%s",
    result.chunks_total,
    result.chunks_selected,
    result.generated,
    result.skipped,
    result.errors,
)
result

In [ ]:
final_wav = Path("/content/out/full.wav")
if not final_wav.exists():
    raise FileNotFoundError("Expected final WAV at /content/out/full.wav")

logger.info("Final WAV ready at %s", final_wav)
display(Audio("/content/out/full.wav"))

---

**Personal use only. AI-generated audio. Do not redistribute generated audio.**